# Project: BBLF AI Selector v2 
# Section: Model Scoring - Expected Points Model
## Sub Section: During Tourny (Post Rnd 1)

Goal: Develop streamline scoring process to predict the players expected points for BBL 15

In [34]:
# pip install --upgrade scikit-learn xgboost interpret

# Prerequistes

In [35]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import joblib

import warnings
warnings.filterwarnings("ignore")

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/python_script/pre-season/'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/post_round_1/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data'
py_data_score_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/scoring'

# Import Model Object

In [36]:
# 1. Load model object
model_obj = joblib.load(os.path.join(directory,'models/ps_all_mdl_1'))

# 2. Model Feature Names
feat_list = model_obj.get_booster().feature_names



# Import Datasets required for Model Scoring

In [37]:
# 1. Import Datasets
# All BBL 15 Player & their Team Data
player_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_efp_rnd_1.csv'), low_memory=False)
player_df = player_df[["Full_Name","player", "Team"]].rename(columns = {"Full_Name":"Name"})

# BBL 15 Player Features
# Prior Season Features (Lags)
lags_15_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_lags.csv'), low_memory=False)
lags_15_df = lags_15_df.drop(["Unnamed: 0"], axis = 1)

# BBL14 Fixture
# Need a table for each team with opposition and where they play against the opposition
team_venue_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
team_venue_df['Game'] = team_venue_df.groupby('Team').cumcount() + 1

# 2. Join Datasets
# Join BBL15 Player Team Data - All Player Fixture Possible Scenarios
player_fix_scen_df = pd.merge(player_df , team_venue_df, left_on = ["Team"], right_on = ["Team"], how = "left")
player_fix_scen_df = player_fix_scen_df.rename(columns = {"Opposition":"opp", "Venue":"venue"})

## Join BBL14 Player Feature Data - All Player Fixture Possible Scenarios
bbl15_scen_df = pd.merge(player_fix_scen_df , lags_15_df, left_on = ["player"], right_on = ["player"], how = "left")
print(bbl15_scen_df)

unique_play_fix = player_fix_scen_df['player'].unique()

unique_lag = lags_15_df['player'].unique()

               Name         player                 Team                opp  \
0     Mark Steketee    MT Steketee      Melbourne Stars  Hobart Hurricanes   
1     Mark Steketee    MT Steketee      Melbourne Stars  Adelaide Strikers   
2     Mark Steketee    MT Steketee      Melbourne Stars      Sydney Sixers   
3     Mark Steketee    MT Steketee      Melbourne Stars     Sydney Thunder   
4     Mark Steketee    MT Steketee      Melbourne Stars      Brisbane Heat   
...             ...            ...                  ...                ...   
1405  Will Salzmann  Will Salzmann  Melbourne Renegades    Perth Scorchers   
1406  Will Salzmann  Will Salzmann  Melbourne Renegades    Melbourne Stars   
1407  Will Salzmann  Will Salzmann  Melbourne Renegades     Sydney Thunder   
1408  Will Salzmann  Will Salzmann  Melbourne Renegades    Perth Scorchers   
1409  Will Salzmann  Will Salzmann  Melbourne Renegades  Adelaide Strikers   

                         venue  Home_f  Round  Game      Full_N

# Override Predictions with Actual Values

In [38]:
# 1. Extract Actual Points Data
bbl15_prior_df = pd.read_csv(os.path.join(add_data_directory,'player_bbl15_pts_rnd_1.csv'), low_memory=False)
bbl15_prior_df = bbl15_prior_df.drop(['Team','Price'], axis = 1)
bbl15_prior_df.columns = bbl15_prior_df.columns.str.replace(r'Game_', '')
bbl15_prior_df = pd.melt(bbl15_prior_df, id_vars=['player'], value_vars= ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10'],
                         var_name='Game', value_name='fantasy_point').dropna()
bbl15_prior_df['Game'] = bbl15_prior_df['Game'].astype(int)
bbl15_prior_df = bbl15_prior_df.rename(columns={"player": "Name"})

# 2. Override Expected Points with Actual Points where available
bbl15_scen_df = pd.merge(bbl15_scen_df, bbl15_prior_df, left_on = ["Name","Game"], right_on = ["Name","Game"], how = "left")

display(bbl15_scen_df.head(10))

,Name,player,Team,opp,venue,Home_f,Round,Game,Full_Name,season_fp_lag1,...,sd_season_dots_lag3,avg_season_extras_lag3,max_season_extras_lag3,min_season_extras_lag3,sd_season_extras_lag3,avg_season_econ_lag3,max_season_econ_lag3,min_season_econ_lag3,sd_season_econ_lag3,fantasy_point
0,Mark Steketee,MT Steketee,Melbourne Stars,Hobart Hurricanes,Melbourne Cricket Ground,1,1,1,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
1,Mark Steketee,MT Steketee,Melbourne Stars,Adelaide Strikers,Adelaide Oval,0,2,2,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
2,Mark Steketee,MT Steketee,Melbourne Stars,Sydney Sixers,SCG,0,3,3,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
3,Mark Steketee,MT Steketee,Melbourne Stars,Sydney Thunder,Other,0,3,4,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
4,Mark Steketee,MT Steketee,Melbourne Stars,Brisbane Heat,GABBA,0,5,5,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
5,Mark Steketee,MT Steketee,Melbourne Stars,Melbourne Renegades,Melbourne Cricket Ground,1,5,6,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
6,Mark Steketee,MT Steketee,Melbourne Stars,Sydney Sixers,Melbourne Cricket Ground,1,6,7,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
7,Mark Steketee,MT Steketee,Melbourne Stars,Melbourne Renegades,Marvel,0,7,8,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
8,Mark Steketee,MT Steketee,Melbourne Stars,Adelaide Strikers,Melbourne Cricket Ground,1,8,9,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN
9,Mark Steketee,MT Steketee,Melbourne Stars,Perth Scorchers,Perth Stadium,0,9,10,Mark Steketee,514.0,...,3.48466,1.857143,6.0,0.0,2.267787,1.857143,6.0,0.0,2.267787,NaN


# Score BBL 15 Data using Model Object

In [39]:
# 1. Score Data
# Create model scoring df by matching train model df columns
bbl15_play_scen_df_model = bbl15_scen_df[feat_list]

# Player Expected Fantasy Point Scoring
bbl15_play_exp_fp = model_obj.predict(bbl15_play_scen_df_model)

bbl15_scen_model_score_full = bbl15_scen_df.copy()
bbl15_scen_model_score_full["mdl_exp_pts"] = bbl15_play_exp_fp.copy()

# Where actual points are available, override expected points with actual points
bbl15_scen_model_score_full["exp_pts"] = np.where(bbl15_scen_model_score_full["fantasy_point"].isna(),
                                                    bbl15_scen_model_score_full["mdl_exp_pts"],
                                                    bbl15_scen_model_score_full["fantasy_point"])
# Flag to indicate if points are actual or expected
bbl15_scen_model_score_full["pts_type"] = np.where(bbl15_scen_model_score_full["fantasy_point"].isna(),
                                                    "exp",
                                                    "actl")

# Save Model Scoring Dataframes

In [40]:
# Short Version - Only Key Columns 
bbl15_scen_model_score_short = bbl15_scen_model_score_full[["player","Name","Team","Round","opp","venue","Home_f","exp_pts","pts_type"]]

# Save Model Scoring DataFrame
bbl15_scen_model_score_full.to_csv(os.path.join(py_data_score_directory,'bbl15_efp_model_score_full_pr1.csv'), index=False)
bbl15_scen_model_score_short.to_csv(os.path.join(py_data_score_directory,'bbl15_efp_model_score_short_pr1.csv'), index=False)